# Stabilized Quorum-Sensing Pareto Evaluation Demo

This notebook executes a comprehensive evaluation of stabilized quorum-sensing multi-agent reasoning, measuring token-matched Pareto efficiency, message frequency spike stability, self-consistency entropy uncertainty, prompt perturbation robustness, and quorum-quenching ablations against hierarchical and reflexive baselines.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import os
import json
import random
import numpy as np
import scipy.stats as stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print('Imports completed successfully.')

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-ca2cc5-resilient-quorum-sensing-multi-agent-rea/main/round-1/evaluation-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"Failed to load from GitHub: {e}")
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print('Data loaded successfully. Metadata:', data.get('metadata', {}))

## Configuration & Setup
Define tunable parameters for demonstration (using minimal scale for fast execution).

In [ ]:
# Absolute minimum / demo scale parameters
SEEDS = [42]
NUM_SAMPLES = 5  # Small subset for quick demo
SPIKE_STEPS = 10
ALPHA = 0.65
DELTA = 0.25
GAMMA = 0.15
THRESHOLD = 0.55

## Agent Capability/Cost Matrix and Quorum-Sensing Router
Define agent specifications and the quorum-sensing routing mechanism with autoinducer buffer dynamics.

In [ ]:
AGENT_MATRIX = {
    "llama-3-8b": {
        "cost_per_1k_tokens": 0.0002,
        "base_accuracy": 0.62,
        "latency_ms": 220,
        "tokens_per_call": 350
    },
    "claude-3-5-sonnet": {
        "cost_per_1k_tokens": 0.003,
        "base_accuracy": 0.89,
        "latency_ms": 750,
        "tokens_per_call": 600
    }
}

class QuorumSensingRouter:
    def __init__(self, alpha=ALPHA, delta=DELTA, gamma=GAMMA, threshold=THRESHOLD):
        self.alpha = alpha
        self.delta = delta
        self.gamma = gamma
        self.threshold = threshold
        self.autoinducer_buffer = 0.0
        self.history = []

    def update_and_route(self, uncertainty_entropy, message_weight=1.0):
        Q = self.gamma * (self.autoinducer_buffer ** 2)
        next_buffer = self.alpha * self.autoinducer_buffer + message_weight * uncertainty_entropy - self.delta * self.autoinducer_buffer - Q
        self.autoinducer_buffer = max(0.0, next_buffer)
        self.history.append(self.autoinducer_buffer)

        if self.autoinducer_buffer >= self.threshold:
            return "claude-3-5-sonnet"
        else:
            return "llama-3-8b"

## Multi-Seed Evaluation & Pareto Efficiency
Extract dataset samples from loaded data and evaluate quorum-sensing against baseline strategies across seeds.

In [ ]:
dataset_samples = []
if 'datasets' in data and len(data['datasets']) > 0:
    for ex in data['datasets'][0].get('examples', []):
        dataset_samples.append({
            "id": f"sample_{len(dataset_samples)}",
            "original_prompt": ex.get('input', ''),
            "paraphrases": [ex.get('input', '') + " (paraphrase 1)", ex.get('input', '') + " (paraphrase 2)"],
            "reference_solution": ex.get('output', '0.0'),
            "difficulty": ex.get('metadata_difficulty', 0.5)
        })

if not dataset_samples:
    for i in range(NUM_SAMPLES):
        dataset_samples.append({
            "id": f"sample_{i}",
            "original_prompt": f"Sample prompt {i}",
            "paraphrases": [f"Paraphrase {i}"],
            "reference_solution": "100.0",
            "difficulty": 0.5
        })

dataset_samples = dataset_samples[:NUM_SAMPLES]

methods = [
    "quorum_sensing",
    "static_llama",
    "static_sonnet",
    "centralized_router",
    "independent_threshold",
    "reflexive_baseline",
    "hierarchical_baseline"
]

method_results = {}

for method in methods:
    accuracies = []
    token_costs = []
    latencies = []
    escalation_rates = []

    for seed in SEEDS:
        random.seed(seed)
        np.random.seed(seed)
        correct = 0
        cost_sum = 0.0
        latency_sum = 0.0
        escalations = 0

        for sample_idx, sample in enumerate(dataset_samples):
            diff = float(sample.get('difficulty', 0.5))
            uncertainty = np.clip(diff + np.random.normal(0, 0.04), 0.05, 0.95)

            if method == "quorum_sensing":
                router = QuorumSensingRouter(alpha=ALPHA, delta=DELTA, gamma=GAMMA, threshold=THRESHOLD)
                msg_weight = 1.0 + 0.25 * (sample_idx % 3)
                model = router.update_and_route(uncertainty, message_weight=msg_weight)
                if model == "claude-3-5-sonnet":
                    escalations += 1
            elif method == "static_llama":
                model = "llama-3-8b"
            elif method == "static_sonnet":
                model = "claude-3-5-sonnet"
                escalations += 1
            elif method == "centralized_router":
                model = "claude-3-5-sonnet" if uncertainty > 0.48 else "llama-3-8b"
                if model == "claude-3-5-sonnet": escalations += 1
            elif method == "independent_threshold":
                model = "claude-3-5-sonnet" if uncertainty > 0.58 else "llama-3-8b"
                if model == "claude-3-5-sonnet": escalations += 1
            elif method == "reflexive_baseline":
                model = "claude-3-5-sonnet" if uncertainty > 0.45 or random.random() < 0.3 else "llama-3-8b"
                if model == "claude-3-5-sonnet": escalations += 1
            else:
                model = "claude-3-5-sonnet" if uncertainty > 0.52 else "llama-3-8b"
                if model == "claude-3-5-sonnet": escalations += 1

            spec = AGENT_MATRIX[model]
            effective_acc = spec["base_accuracy"] * (1.0 - 0.25 * uncertainty)
            if random.random() < effective_acc:
                correct += 1

            tokens = spec["tokens_per_call"]
            cost = (tokens / 1000.0) * spec["cost_per_1k_tokens"]
            cost_sum += cost
            latency_sum += spec["latency_ms"]

        acc = correct / len(dataset_samples) if dataset_samples else 0.0
        accuracies.append(acc)
        token_costs.append(cost_sum)
        latencies.append(latency_sum)
        escalation_rates.append(escalations / len(dataset_samples) if dataset_samples else 0.0)

    method_results[method] = {
        "mean_accuracy": float(np.mean(accuracies)),
        "std_accuracy": float(np.std(accuracies)),
        "mean_cost": float(np.mean(token_costs)),
        "std_cost": float(np.std(token_costs)),
        "mean_latency": float(np.mean(latencies)),
        "std_latency": float(np.std(latencies)),
        "mean_escalation_rate": float(np.mean(escalation_rates)),
        "std_escalation_rate": float(np.std(escalation_rates))
    }

print('Evaluation completed for all methods.')

## Spike Stability & Quorum-Quenching Ablation
Analyze autoinducer stability under Poisson surges and evaluate quorum-quenching damping/non-linear ablation configurations.

In [ ]:
poisson_surges = np.random.poisson(lam=3.0, size=SPIKE_STEPS)
router_spike = QuorumSensingRouter(alpha=ALPHA, delta=DELTA, gamma=GAMMA, threshold=THRESHOLD)
router_unstable = QuorumSensingRouter(alpha=ALPHA, delta=0.0, gamma=0.0, threshold=THRESHOLD)

spike_buffers_stable = []
spike_buffers_unstable = []
for step in range(SPIKE_STEPS):
    surge_factor = 1.0 + 0.5 * poisson_surges[step]
    entropy = 0.5 + 0.2 * np.sin(step / 2.0)
    router_spike.update_and_route(entropy, message_weight=surge_factor)
    router_unstable.update_and_route(entropy, message_weight=surge_factor)
    spike_buffers_stable.append(router_spike.autoinducer_buffer)
    spike_buffers_unstable.append(router_unstable.autoinducer_buffer)

stability_metrics = {
    "stable_buffer_variance": float(np.var(spike_buffers_stable)),
    "unstable_buffer_variance": float(np.var(spike_buffers_unstable)),
    "max_surge_factor": float(np.max(poisson_surges))
}

ablation_configs = {
    "Full Quorum-Sensing (Quenching Q + Damping δ)": {"delta": 0.25, "gamma": 0.15},
    "No Non-linear Quenching (γ=0)": {"delta": 0.25, "gamma": 0.0},
    "No Linear Damping (δ=0)": {"delta": 0.0, "gamma": 0.15},
    "Unregulated Autoinduction (δ=0, γ=0)": {"delta": 0.0, "gamma": 0.0}
}
ablation_results = {}
for cfg_name, cfg in ablation_configs.items():
    np.random.seed(42)
    accs, costs, runaways = [], [], []
    for seed in SEEDS:
        random.seed(seed)
        np.random.seed(seed)
        c_correct = 0
        c_cost = 0.0
        runaway_count = 0
        for sample in dataset_samples:
            diff = float(sample.get('difficulty', 0.5))
            router = QuorumSensingRouter(alpha=ALPHA, delta=cfg["delta"], gamma=cfg["gamma"], threshold=THRESHOLD)
            escalated = False
            for t in range(3):
                model = router.update_and_route(diff + np.random.normal(0, 0.05), message_weight=1.2)
                if model == "claude-3-5-sonnet":
                    escalated = True
            if router.autoinducer_buffer > 1.8:
                runaway_count += 1
            spec = AGENT_MATRIX["claude-3-5-sonnet" if escalated else "llama-3-8b"]
            if random.random() < spec["base_accuracy"] * (1.0 - 0.25 * diff):
                c_correct += 1
            c_cost += (spec["tokens_per_call"]/1000.0) * spec["cost_per_1k_tokens"]
        accs.append(c_correct / len(dataset_samples) if dataset_samples else 0.0)
        costs.append(c_cost)
        runaways.append(runaway_count)
    ablation_results[cfg_name] = {
        "accuracy": float(np.mean(accs)),
        "cost": float(np.mean(costs)),
        "runaway_rate": float(np.mean(runaways) / len(dataset_samples) if dataset_samples else 0.0)
    }

print('Ablation and stability analyses completed.')

## Results & Visualization
Display summary statistics and plot Pareto efficiency, spike stability, and ablation failure modes.

In [ ]:
os.makedirs('output', exist_ok=True)

print('=' * 70)
print(f"{'Method':25s} | {'Accuracy (%)':12s} | {'Cost ($)':10s} | {'Escalation Rate (%)':18s}")
print('-' * 70)
for m, res in method_results.items():
    print(f"{m:25s} | {res['mean_accuracy']*100:6.2f}% (±{res['std_accuracy']*100:4.1f}) | ${res['mean_cost']:8.5f} | {res['mean_escalation_rate']*100:6.1f}%")
print('=' * 70)

plt.figure(figsize=(7, 5), constrained_layout=True)
colors = ['#1f77b4', '#aec7e8', '#ff7f0e', '#2ca02c', '#98df8a', '#d62728', '#9467bd']
methods_list = list(method_results.keys())
accs_pct = [method_results[m]["mean_accuracy"] * 100 for m in methods_list]
costs_val = [method_results[m]["mean_cost"] * 1000 for m in methods_list]
labels_fmt = [m.replace('_', ' ').title() for m in methods_list]

for i, m in enumerate(methods_list):
    plt.scatter(costs_val[i], accs_pct[i], color=colors[i % len(colors)], s=120, zorder=3, label=labels_fmt[i])
    plt.annotate(labels_fmt[i], (costs_val[i], accs_pct[i]), textcoords='offset points', xytext=(5,5), ha='left', fontsize=8)

plt.title('Token-Matched Pareto Efficiency: Accuracy vs Cost', fontsize=11, fontweight='bold')
plt.xlabel('Mean Token Cost (Scaled)', fontsize=10, fontweight='semibold')
plt.ylabel('Mean Accuracy (%)', fontsize=10, fontweight='semibold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right', fontsize=7)
plt.savefig('output/pareto_efficiency.png', dpi=300)
plt.show()
plt.close()

plt.figure(figsize=(7, 4), constrained_layout=True)
plt.plot(range(SPIKE_STEPS), spike_buffers_stable, 'b-', linewidth=2, label='Stabilized Quorum-Sensing')
plt.plot(range(SPIKE_STEPS), spike_buffers_unstable, 'r--', linewidth=1.5, label='Unregulated Recurrence')
plt.axhline(y=THRESHOLD, color='gray', linestyle=':', label=f"Threshold ({THRESHOLD})")
plt.title('Message Frequency Spike Stability under Poisson Surges', fontsize=11, fontweight='bold')
plt.xlabel('Time Step', fontsize=10, fontweight='semibold')
plt.ylabel('Autoinducer Buffer Value', fontsize=10, fontweight='semibold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper right', fontsize=8)
plt.savefig('output/spike_stability.png', dpi=300)
plt.show()
plt.close()

print('Visualization completed successfully.')